<a href="https://colab.research.google.com/github/asiddiqha/deep-learning-transfer-learning/blob/main/Transfer_Learning_Mammal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mammal Image Classification with Transfer Learning

**Model:** MobileNetV2 (ImageNet)  
**Task:** 45-class mammal image classification  
**Approach:** Transfer learning + fine-tuning of the final 20 backbone layers

This notebook demonstrates an end-to-end deep learning workflow: dataset preparation, image generators, pretrained CNN feature extraction, fine-tuning, evaluation, model persistence, and single-image inference.

## 1. Dataset

The notebook uses the [Mammals Image Classification Dataset – 45 Animals](https://www.kaggle.com/datasets/asaniczka/mammals-image-classification-dataset-45-animals) from Kaggle. The original run produced **13,751 images across 45 classes**, split into 11,020 training images and 2,731 validation images using an 80/20 split.

In [1]:
# Download and extract the dataset in Google Colab
!mkdir -p mammals_dataset
!kaggle datasets download -d asaniczka/mammals-image-classification-dataset-45-animals -p mammals_dataset
!unzip -q mammals_dataset/mammals-image-classification-dataset-45-animals.zip -d mammals_dataset

Dataset URL: https://www.kaggle.com/datasets/asaniczka/mammals-image-classification-dataset-45-animals
License(s): ODC Attribution License (ODC-By)
100% 185M/185M [00:01<00:00, 104MB/s]



## 2. Imports and configuration

In [2]:
import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 45
DATA_DIR = "mammals_dataset/mammals"
MODEL_PATH = "mammals_model.keras"

print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

TensorFlow: 2.20.0
GPU available: True


## 3. Training and validation data

An 80/20 validation split is created from the directory structure. Images are resized to 224×224 and scaled to the [0, 1] range.

In [3]:
datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.20
)

train_data = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

val_data = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

print(f"Training images: {train_data.samples}")
print(f"Validation images: {val_data.samples}")
print(f"Classes: {train_data.num_classes}")

Found 11020 images belonging to 45 classes.
Found 2731 images belonging to 45 classes.
Training images: 11020
Validation images: 2731
Classes: 45


## 4. Build the MobileNetV2 transfer-learning model

The ImageNet-trained MobileNetV2 backbone is initially frozen. A lightweight classification head is added for the 45 mammal classes.

In [4]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(*IMG_SIZE, 3)
)
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation="relu")(x)
outputs = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=outputs)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,597,485 (9.91 MB)

 Trainable params: 339,501 (1.30 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 5. Fine-tune the final 20 backbone layers

After establishing the transfer-learning architecture, the last 20 MobileNetV2 layers are made trainable while earlier layers remain frozen. This allows the model to adapt higher-level visual features to the mammal classification task while retaining most pretrained representations.

In [5]:
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

trainable_layers = sum(layer.trainable for layer in base_model.layers)
frozen_layers = len(base_model.layers) - trainable_layers

print(f"Frozen backbone layers: {frozen_layers}")
print(f"Trainable backbone layers: {trainable_layers}")

# Re-compile after changing trainable status.
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

Frozen backbone layers: 134
Trainable backbone layers: 20


## 6. Train the model

In [6]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 76s 152ms/step - accuracy: 0.7559 - loss: 1.0459 - val_accuracy: 0.8502 - val_loss: 0.5161
Epoch 2/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 25s 72ms/step - accuracy: 0.9318 - loss: 0.2500 - val_accuracy: 0.8850 - val_loss: 0.3940
Epoch 3/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 23s 68ms/step - accuracy: 0.9718 - loss: 0.1149 - val_accuracy: 0.9041 - val_loss: 0.3433
Epoch 4/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 24s 69ms/step - accuracy: 0.9888 - loss: 0.0587 - val_accuracy: 0.9088 - val_loss: 0.3231
Epoch 5/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 24s 68ms/step - accuracy: 0.9926 - loss: 0.0406 - val_accuracy: 0.9099 - val_loss: 0.3242
Epoch 6/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 22s 64ms/step - accuracy: 0.9949 - loss: 0.0283 - val_accuracy: 0.9213 - val_loss: 0.3138
Epoch 7/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 23s 68ms/step - accuracy: 0.9960 - loss: 0.0238 - val_accuracy: 0.9202 - val_loss: 0.2971
Epoch 8/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 24s 70ms/step - accuracy: 0.9967 - loss: 0.0181 -

## 7. Training results

The original notebook run achieved **99.68% training accuracy** and **91.58% validation accuracy** after 10 epochs.

In [7]:
print(f"Final training accuracy:   {history.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"Final validation loss:     {history.history['val_loss'][-1]:.4f}")

Final training accuracy:   0.9968
Final validation accuracy: 0.9158
Final validation loss:     0.3584


## 8. Class mapping

In [8]:
class_indices = train_data.class_indices
class_names = [name for name, index in sorted(class_indices.items(), key=lambda item: item[1])]

print("Number of classes:", len(class_names))
print(class_names)

Number of classes: 45
['african_elephant', 'alpaca', 'american_bison', 'anteater', 'arctic_fox', 'armadillo', 'baboon', 'badger', 'blue_whale', 'brown_bear', 'camel', 'dolphin', 'giraffe', 'groundhog', 'highland_cattle', 'horse', 'jackal', 'kangaroo', 'koala', 'manatee', 'mongoose', 'mountain_goat', 'opossum', 'orangutan', 'otter', 'polar_bear', 'porcupine', 'red_panda', 'rhinoceros', 'sea_lion', 'seal', 'snow_leopard', 'squirrel', 'sugar_glider', 'tapir', 'vampire_bat', 'vicuna', 'walrus', 'warthog', 'water_buffalo', 'weasel', 'wildebeest', 'wombat', 'yak', 'zebra']


## 9. Save the trained model

The native Keras `.keras` format is used for the polished version rather than the legacy HDF5 format.

In [9]:
model.save(MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")

Model saved to: mammals_model.keras


## 10. Single-image inference

Place a test image at `TEST_IMAGE_PATH` and run the cell below. The model returns the predicted mammal class and confidence.

In [11]:
TEST_IMAGE_PATH = "/content/Koala.jpg"

if not os.path.exists(TEST_IMAGE_PATH):
    print(f"Test image not found: {TEST_IMAGE_PATH}")
else:
    loaded_model = load_model(MODEL_PATH)

    img = image.load_img(TEST_IMAGE_PATH, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    prediction = loaded_model.predict(img_array, verbose=0)[0]
    predicted_index = int(np.argmax(prediction))
    predicted_class = class_names[predicted_index]
    confidence = float(prediction[predicted_index])

    print(f"Predicted mammal: {predicted_class}")
    print(f"Confidence: {confidence:.2%}")

Predicted mammal: koala
Confidence: 99.99%
